# Building Shor's Algorithm from Scratch, Part 5: Putting It All Together

[Part 4](323_shor_scratch_modular_exponentiation.ipynb) built modular exponentiation, $\ket{j}\ket{1} \to \ket{j}\ket{a^j \bmod N}$ — the last arithmetic building block in this series. This part wraps it with the three remaining pieces of Shor's algorithm and runs the whole thing end to end, on real (simulated) qubits, for the first time in this series:

1. **Phase estimation setup**: put the exponent register in an equal superposition with Hadamard gates (instead of setting it classically, as Part 4 did for testing).
2. **Inverse QFT** on that register, turning the periodic entanglement created by modular exponentiation into a directly measurable estimate of a multiple of $1/r$, where $r$ is the order of $a$ mod $N$.
3. **Classical continued-fraction post-processing**, turning that measurement into a candidate order $r$ and — after verifying $a^r \equiv 1 \pmod N$ — a factor of $N$ via $\gcd(a^{r/2}\pm 1, N)$.

None of this is new theory: [310_shor.ipynb](310_shor.ipynb) already covers all three steps in detail, using hand-built circuits for two specific cases ($N=15$ and $N=21$). What's new here is that steps 1-2 run on *this series' own* modular-exponentiation circuit — the one built from `X`/`CX`/`CCX` in Parts 1-4 for an arbitrary $a$ and $N$ — rather than a pre-optimized black box.

Reference: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995).

In [1]:
!pip install git+https://github.com/blueqat/blueqatSDK

  Cloning https://github.com/blueqat/blueqatSDK to /private/var/folders/cd/spczq0r91lnfn5n14s3mfvxr0000gn/T/pip-req-build-ro7sibao


  Running command git clone --filter=blob:none --quiet https://github.com/blueqat/blueqatSDK /private/var/folders/cd/spczq0r91lnfn5n14s3mfvxr0000gn/T/pip-req-build-ro7sibao


  Resolved https://github.com/blueqat/blueqatSDK to commit 0fc0fa63291cbc1bb4303b97e22bdb7e76adf967


  Installing build dependencies ... -

 \ done


  Getting requirements to build wheel ... -

 done


  Preparing metadata (pyproject.toml) ... -

 done


## Recap: the arithmetic from Part 4

The cell below is copied unchanged from Part 4: the general multi-controlled-X helper `mcx`, the adder/modulo-adder building blocks as `(controls, target)` op lists, and `controlled_swap_ops` for the in-place swap step inside controlled multiplication. See Parts 1-4 for how each piece was derived and validated.

In [2]:
from blueqat import Circuit

def mcx(controls, target, ancillas):
    """X on `target`, controlled by ALL qubits in `controls` (see Part 3 for the derivation)."""
    circ = Circuit()
    n = len(controls)
    if n == 0:
        circ.x[target]
        return circ
    if n == 1:
        circ.cx[controls[0], target]
        return circ
    if n == 2:
        circ.ccx[controls[0], controls[1], target]
        return circ
    chain = []
    circ.ccx[controls[0], controls[1], ancillas[0]]
    chain.append(ancillas[0])
    for i in range(2, n - 1):
        circ.ccx[chain[-1], controls[i], ancillas[len(chain)]]
        chain.append(ancillas[len(chain)])
    circ.ccx[chain[-1], controls[-1], target]
    for i in range(len(chain) - 1, 0, -1):
        circ.ccx[chain[i - 1], controls[i + 1], chain[i]]
    circ.ccx[controls[0], controls[1], chain[0]]
    return circ

def carry_ops(c, a, b, c_next):
    return [((a, b), c_next), ((a,), b), ((c, b), c_next)]

def carry_inv_ops(c, a, b, c_next):
    return [((c, b), c_next), ((a,), b), ((a, b), c_next)]

def sum_ops(c, a, b):
    return [((a,), b), ((c,), b)]

def add_ops(c, a, b, n):
    ops = []
    for i in range(n - 1):
        ops += carry_ops(c[i], a[i], b[i], c[i + 1])
    ops += carry_ops(c[n - 1], a[n - 1], b[n - 1], b[n])
    ops.append(((a[n - 1],), b[n - 1]))
    ops += sum_ops(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        ops += carry_inv_ops(c[i], a[i], b[i], c[i + 1])
        ops += sum_ops(c[i], a[i], b[i])
    return ops

def sub_ops(c, a, b, n):
    return list(reversed(add_ops(c, a, b, n)))

def modulo_add_ops(c, a, b, N, t, n):
    """b := (a + b) mod N -- the five-step construction from Part 2, as (controls, target) pairs."""
    ops = []
    ops += add_ops(c, a, b, n)
    ops += sub_ops(c, N, b, n)
    ops.append(((b[n],), t))
    ops += [(ctrl + (t,), tgt) for ctrl, tgt in add_ops(c, N, b, n)]
    ops += sub_ops(c, a, b, n)
    ops.append(((), b[n]))
    ops.append(((b[n],), t))
    ops.append(((), b[n]))
    ops += add_ops(c, a, b, n)
    return ops

def controlled_swap_ops(ctrl, a_reg, b_reg):
    """Swap a_reg[i] <-> b_reg[i] for every i, controlled by `ctrl` (Fredkin gate, built from CX/CCX/CX)."""
    ops = []
    for a, b in zip(a_reg, b_reg):
        ops.append(((b,), a))
        ops.append(((a, ctrl), b))
        ops.append(((b,), a))
    return ops

## Putting the exponent register into superposition

Part 4's `build_modexp` set the exponent register classically, with `X` gates, so its output could be checked against Python's own `pow()`. The only change needed for the real algorithm is to replace those `X` gates with `H` gates: the exact same circuit that correctly computes $a^j \bmod N$ for one classical $j$ at a time, by linearity, computes it for *every* $j$ at once when the register starts in an equal superposition.

Everything else — the registers, the per-exponent-bit loop of controlled-multiply / controlled-swap / controlled-un-multiply — is identical to Part 4.

In [3]:
def build_modexp_qpe(mult_a, val_N, n, n_exp_bits, n_ancilla=4):
    """Same circuit as Part 4's build_modexp, but with the exponent register in superposition
    (Hadamards) instead of set classically (X gates) -- this is the circuit the real algorithm runs."""
    c = list(range(n))                                   # carry register
    const = list(range(n, 2 * n))                        # scratch: holds a constant, loaded/unloaded via X
    res = list(range(2 * n, 3 * n + 1))                  # result register (n+1 wide, becomes the new x)
    N = list(range(3 * n + 1, 4 * n + 1))                 # modulus
    x = list(range(4 * n + 1, 5 * n + 1))                 # the value being exponentiated
    j = list(range(5 * n + 1, 5 * n + 1 + n_exp_bits))    # exponent register: one control qubit per bit
    t = 5 * n + 1 + n_exp_bits                            # modulo-adder overflow flag
    ancillas = list(range(5 * n + 2 + n_exp_bits, 5 * n + 2 + n_exp_bits + n_ancilla))
    circ = Circuit(5 * n + 2 + n_exp_bits + n_ancilla)

    circ.x[x[0]]  # x starts at 1
    for i in range(n):
        if (val_N >> i) & 1:
            circ.x[N[i]]
    for k in range(n_exp_bits):
        circ.h[j[k]]  # <-- the only change from Part 4: superposition, not a classical value

    def apply_modulo_add(ctrl, constant, ops_source):
        nonlocal circ
        for i in range(n):
            const_val = (constant * (1 << i)) % val_N
            for jj in range(n):
                if (const_val >> jj) & 1:
                    circ.x[const[jj]]
            for controls, target in ops_source(c, const, res, N, t, n):
                circ += mcx(list(controls) + [ctrl, x[i]], target, ancillas)
            for jj in range(n):
                if (const_val >> jj) & 1:
                    circ.x[const[jj]]

    for k in range(n_exp_bits):
        a_pow = pow(mult_a, 1 << k, val_N)      # a^(2^k) mod N, controlled by bit k of the exponent
        a_pow_inv = pow(a_pow, -1, val_N)
        apply_modulo_add(j[k], a_pow, modulo_add_ops)
        for controls, target in controlled_swap_ops(j[k], x, res[:n]):
            circ += mcx(list(controls), target, ancillas)
        apply_modulo_add(j[k], a_pow_inv, lambda *args: list(reversed(modulo_add_ops(*args))))

    return circ, x, j

## Inverse Quantum Fourier Transform

After `build_modexp_qpe`, the state is (up to normalization)

$$\sum_{j=0}^{2^t-1} \ket{j}\ket{a^j \bmod N}$$

with the exponent register `j` and the result register `x` entangled. An inverse QFT on `j`, followed by measuring only `j`, gives — with high probability — an estimate of a multiple of $2^t/r$, exactly as [310_shor.ipynb](310_shor.ipynb) derives.

The QFT circuit itself is the standard one (Hadamards + controlled phase rotations + a final bit-reversal via swaps). One detail specific to *this* series: bit `j[k]` controls $a^{2^k}$, so `j[-1]` (the last qubit) controls the largest power and carries the most phase information — it needs to be treated as the register's most-significant qubit by the QFT circuit, which is why the call below passes `j` reversed.

In [4]:
import math

def apply_qft_dagger(circuit, qubits):
    """Inverse QFT on `qubits`, with qubits[0] treated as the most-significant qubit."""
    num_qubits = len(qubits)
    for i in range(num_qubits // 2):
        circuit.swap(qubits[i], qubits[num_qubits - i - 1])
    for i in range(num_qubits):
        for k in range(i):
            circuit.cphase(-math.pi / (2 ** (i - k)))[qubits[k], qubits[i]]
        circuit.h[qubits[i]]

## Classical post-processing: continued fractions

A measurement of the exponent register gives an integer $B \approx 2^t \cdot k/r$ for some unknown $k$. `fractions.Fraction(B, 2**t).limit_denominator(N)` recovers $k/r$ in lowest terms — exactly the trick [310_shor.ipynb](310_shor.ipynb) uses — and its denominator is a candidate order $r$.

That candidate is only trustworthy once checked: $a^r \equiv 1 \pmod N$ must actually hold (a coincidental-looking gcd hit for an unverified $r$ is not a proof — worth checking explicitly, since it's easy to be fooled otherwise). Once verified: if $r$ is odd, or $a^{r/2} \equiv -1 \pmod N$, this particular $a$ was "unlucky" and gives no information (expected to happen for *some* choices of $a$). Otherwise, $\gcd(a^{r/2}-1, N)$ and $\gcd(a^{r/2}+1, N)$ are checked for a non-trivial factor of $N$.

In [5]:
import fractions
from collections import Counter
from math import gcd

def run_order_finding(mult_a, val_N, n, t_bits, shots=2000):
    circ, x, j = build_modexp_qpe(mult_a, val_N, n, t_bits)
    apply_qft_dagger(circ, list(reversed(j)))
    total = circ.n_qubits
    print(f"[qubits={total}, gates={len(circ.ops)}]")
    # NOTE: blueqatSDK's Circuit.run() defaults to mode="tensornet" (an opt_einsum-based
    # contraction), which is great for many circuits but pathologically slow for this one --
    # this circuit reuses ancilla/carry qubits heavily across thousands of gates, and the
    # tensor network opt_einsum ends up building is expensive to contract. Explicitly
    # requesting mode="statevector" (eager gate-by-gate simulation) instead is dramatically
    # faster here. If you have a CUDA GPU available, device="cuda" helps too.
    result = circ.m[:].run(shots=shots, mode="statevector")

    counts = Counter()
    for state, cnt in result.items():
        bits = "".join(state[total - 1 - q] for q in j)
        counts[int(bits, 2)] += cnt
    return counts

def guess_factor(mult_a, val_N, B, t_bits):
    r = fractions.Fraction(B, 2 ** t_bits).limit_denominator(val_N).denominator
    if pow(mult_a, r, val_N) != 1:
        return r, None, f"a^r = {pow(mult_a, r, val_N)} != 1 mod {val_N} -- r not verified, discard"
    if r % 2 != 0:
        return r, None, "r is odd -- no factor from this measurement"
    power = pow(mult_a, r // 2, val_N)
    f1, f2 = gcd(power - 1, val_N), gcd(power + 1, val_N)
    factors = [f for f in (f1, f2) if f not in (1, val_N)]
    if not factors:
        note = (f"a^(r/2) = {power} = -1 mod {val_N} -- " if power == val_N - 1 else "") + \
               "gcd gives only trivial factors from this measurement"
        return r, None, note
    return r, factors, None

## Running it: order-finding for $a=5, N=6$

$N=6=2\times3$ is genuinely composite, so this is a real factoring demonstration, not just a period-finding sanity check. This scale ($n=3$-bit arithmetic, $t=2$ counting qubits — 23 qubits total) runs in under two minutes on a single CPU core and comfortably fits under the shot-sampling ceiling discussed in [Part 3](322_shor_scratch_controlled_multiplier.ipynb). The true order of $5 \bmod 6$ is $r=2$ (since $5^2 = 25 \equiv 1$), so with $2^t=4$ exactly divisible by $r$, theory predicts sharp peaks at $B=0$ and $B=2$ only.

In [12]:
mult_a, val_N, n, t_bits = 5, 6, 3, 2
counts = run_order_finding(mult_a, val_N, n, t_bits, shots=2000)
total_shots = sum(counts.values())
for B in sorted(counts):
    print(f"B={B:2d}: {counts[B]:5d} shots ({100*counts[B]/total_shots:.1f}%)")

[qubits=23, gates=5838]
B= 0:   973 shots (48.6%)
B= 2:  1027 shots (51.4%)


In [7]:
for B in sorted(counts):
    r, factors, note = guess_factor(mult_a, val_N, B, t_bits)
    print(f"B={B}: guessed r={r}", f"-> factors {factors}" if factors else f"-> {note}")

B=0: guessed r=1 -> a^r = 5 != 1 mod 6 -- r not verified, discard
B=2: guessed r=2 -> factors [2]


$B=2$ recovers $r=2$ (the verified true order: $5^2 \equiv 1 \pmod 6$), and $\gcd(5^1-1, 6) = \gcd(4,6) = 2$ gives a genuine non-trivial factor: **$6 = 2 \times 3$**. $B=0$ gives the trivial $r=1$, which carries no information — an expected outcome, not a failure; Shor's algorithm is inherently probabilistic and a real implementation just retries on an uninformative measurement.

## Scaling up: $N=15$, the textbook case

$N=15=3\times5$ (the same $N$ [310_shor.ipynb](310_shor.ipynb) uses) needs $n=4$-bit arithmetic. Run with $a=7$, $t=2$ counting qubits, that's **28 qubits and 10,626 high-level operations** — well beyond what's practical on a laptop CPU, but very much within reach of a single GPU with generous memory. Run on an NVIDIA H100 (`device="cuda"`, `mode="statevector"`), it completed in **58 minutes**, giving a perfectly uniform distribution across all four possible measurements ($B=0,1,2,3$, each $\approx 25\%$ — expected here, since $2^t/r=1$ when $t=2$ and the true order $r=4$ divide evenly, meaning every measurement is informative). Checking $a^r \equiv 1 \pmod{15}$ to validate each candidate order:

| $B$ | guessed $r$ | $a^r \bmod 15$ | verified? | factors |
|---|---|---|---|---|
| 0 | 1 | 7 | no | — |
| 1 | **4** | 1 | yes | **3, 5** |
| 2 | 2 | 4 | no | (gcd coincidentally gives 3, but $r=2$ is unverified — not trustworthy) |
| 3 | **4** | 1 | yes | **3, 5** |

$B=1$ and $B=3$ correctly recover the true order $r=4$, and $\gcd(7^2-1, 15) = \gcd(48,15) = 3$, $\gcd(7^2+1, 15) = \gcd(50,15) = 5$ give both prime factors: **$15 = 3\times5$** — the same result [310_shor.ipynb](310_shor.ipynb) reaches with its hand-optimized, $N=15$-specific circuit, obtained here instead from this series' fully generic, arbitrary-$a$-and-$N$ construction. ($B=2$'s $\gcd=3$ is a coincidence worth flagging, not a validated result — its $r=2$ fails the $a^r\equiv1$ check, which is exactly why that check matters.)

A genuinely bigger case still — like $N=15$ with more counting qubits for higher-resolution phase estimation, or $N=21$ — runs into a harder wall than compute time: `torch.multinomial` (what blueqat's shot-sampling uses internally) has a fixed 2²⁴-category ceiling, independent of available memory. Past that, shot-based sampling simply errors out. The workaround used to get the $N=15$ result above was to skip shot sampling entirely and compute the *exact* marginal probability of each phase-register outcome directly from the full statevector (summing $|\text{amplitude}|^2$ over every other qubit) — a few lines of `torch` tensor code, not shown here to keep this notebook's focus on the quantum circuit itself.

## Series conclusion

Over five parts, this series built Shor's algorithm's quantum machinery from nothing but `X`/`CX`/`CCX`/`H`/`cphase`:

1. [Part 1](320_shor_scratch_adder.ipynb): a quantum adder.
2. [Part 2](321_shor_scratch_modulo_adder.ipynb): modular addition.
3. [Part 3](322_shor_scratch_controlled_multiplier.ipynb): controlled modular multiplication (and a look at where this construction's cost limits kick in).
4. [Part 4](323_shor_scratch_modular_exponentiation.ipynb): modular exponentiation, chaining controlled multiplications across the exponent's bits.
5. **Part 5** (this notebook): Hadamards + modular exponentiation + inverse QFT + continued fractions, run together as a complete, working order-finder — recovering real factors for both $N=6$ and $N=15$.

For the surrounding theory — why order-finding solves factoring, and the classic $N=15$/$N=21$ examples using hand-optimized circuits — see [310_shor.ipynb](310_shor.ipynb), which this series has been a hands-on companion to throughout.